# DeepSeek-OCR × CUAD — Multi-Page A4 Pipeline (5× / 10×)

Extends `DeepSeek_OCR_CUAD_5X10X_Colab.ipynb` with:
1. `.env` loading from `notebooks/.env` (keys outside the repo).
2. **Multi-page A4 rendering** — every long document is split into A4 pages before OCR; no more 1:100 scrolls.
3. **No-training MemSlot** variant — random-init slots, untrained; compared head-to-head with the trained one.
4. **More baselines** — `random` (word drop) and `lead` (truncation) as lower bounds.
5. **Ablation: text→render→OCR** — `summary-ocr` and `prune-ocr` feed the pruned / summarized text through the OCR pipeline. If MemSlot-OCR only wins because of the OCR step, these baselines catch it; if MemSlot saliency is the real driver, they fall behind.
6. **QA reader** = `qwen-long` (10M-token context) so the `full` baseline is never silently truncated.
7. Smaller data: 50 % train subset + 8 test docs × 4 QAs — keeps the full matrix under ~25 min while preserving the evaluation protocol.


## 0. Environment

Priority: Colab Secrets → `notebooks/.env` → shell env. `.env` is git-ignored.

In [ ]:
import os
from pathlib import Path

_KEYS = ('DEEPSEEK_API_KEY', 'DASHSCOPE_API_KEY', 'OPENAI_API_KEY')

try:
    from google.colab import userdata
    for k in _KEYS:
        try:
            v = userdata.get(k)
            if v and not os.environ.get(k):
                os.environ[k] = v
        except Exception:
            pass
except Exception:
    pass

for p in (Path.cwd() / '.env',
          Path.cwd() / 'notebooks' / '.env',
          Path('/content/Distorted-OCR-Compression/notebooks/.env')):
    if p.is_file():
        for line in p.read_text().splitlines():
            line = line.strip()
            if not line or line.startswith('#') or '=' not in line:
                continue
            k, v = line.split('=', 1)
            k = k.strip(); v = v.strip().strip('"').strip("'")
            if k in _KEYS and v and not os.environ.get(k):
                os.environ[k] = v
        print(f'loaded .env from {p}')
        break
else:
    print('no .env found — relying on Colab Secrets or shell env')

assert any(os.environ.get(k) for k in _KEYS), \
    'Set at least one of DEEPSEEK_API_KEY / DASHSCOPE_API_KEY / OPENAI_API_KEY'
print('keys present:', [k for k in _KEYS if os.environ.get(k)])

!nvidia-smi | head -6 || echo 'no GPU detected — DeepSeek-OCR needs A100/H100/L4/A10G'


In [ ]:
%%bash
set -e
pip -q install 'transformers==4.46.3' 'tokenizers==0.20.3' 'accelerate>=0.34,<1.1' einops addict easydict
pip -q install Pillow matplotlib seaborn pandas requests tqdm


In [ ]:
import os, sys
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules or (Path('/content').exists() and os.environ.get('COLAB_RELEASE_TAG'))

if IS_COLAB:
    REPO_ROOT = Path('/content/Distorted-OCR-Compression')
    os.chdir('/content')
    if not REPO_ROOT.exists():
        os.system('git clone https://github.com/KuiyiGao/Distorted-OCR-Compression.git')
    else:
        os.system(f'cd {REPO_ROOT} && git pull')
    os.chdir(REPO_ROOT)
    WORKDIR = Path('/content')
else:
    _here = Path.cwd().resolve()
    for _c in (_here, *_here.parents):
        if (_c / 'deepseek_pipeline').is_dir() and (_c / 'render').is_dir():
            REPO_ROOT = _c; break
    else:
        raise RuntimeError(f'Could not find repo root from {_here}')
    os.chdir(REPO_ROOT)
    WORKDIR = REPO_ROOT / 'runs'
    WORKDIR.mkdir(exist_ok=True)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f'IS_COLAB={IS_COLAB}  REPO_ROOT={REPO_ROOT}  WORKDIR={WORKDIR}')


## 1. CUAD split — 50 % train, 8 test docs × 4 QAs

`random.seed(0)` pins the document order identically to the previous notebook.
We down-sample training to 50 % (≈ 245 contracts) and cap MemSlot training to
the first 120 of those to keep the bf16 training pass under a few minutes on an
A100.

In [ ]:
import json, random, urllib.request, zipfile, io
from pathlib import Path

random.seed(0)
N_TEST_DOCS = 8
K_ANS, K_UNANS = 2, 2
TRAIN_FRACTION = 0.5
TRAIN_CAP_FOR_MEMSLOT = 120

CUAD_PATH = Path('render/data/CUADv1.json')
if not CUAD_PATH.exists():
    CUAD_PATH.parent.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen('https://raw.githubusercontent.com/TheAtticusProject/cuad/main/data.zip') as r:
        with zipfile.ZipFile(io.BytesIO(r.read())) as z:
            with z.open('CUADv1.json') as src, open(CUAD_PATH, 'wb') as dst:
                dst.write(src.read())

cuad = json.load(open(CUAD_PATH))['data']
random.shuffle(cuad)
test_docs = cuad[:N_TEST_DOCS]
remaining = cuad[N_TEST_DOCS:]
train_docs = remaining[: int(len(remaining) * TRAIN_FRACTION)]
train_contexts = [p['context'] for d in train_docs for p in d['paragraphs']]

test_items = []
rng = random.Random(1)
for doc in test_docs:
    for para in doc['paragraphs']:
        ctx = para['context']
        ans = [q for q in para['qas'] if q.get('answers')]
        un  = [q for q in para['qas'] if not q.get('answers')]
        rng.shuffle(ans); rng.shuffle(un)
        for qa in ans[:K_ANS]:
            test_items.append({'qa_id': qa['id'], 'question': qa['question'],
                               'context': ctx, 'gold': [a['text'] for a in qa['answers']],
                               'answerable': True})
        for qa in un[:K_UNANS]:
            test_items.append({'qa_id': qa['id'], 'question': qa['question'],
                               'context': ctx, 'gold': [],
                               'answerable': False})

n_ans = sum(1 for t in test_items if t['answerable'])
print(f'train : {len(train_docs)} docs / {len(train_contexts)} contexts  (cap for MemSlot: {TRAIN_CAP_FOR_MEMSLOT})')
print(f'test  : {len(test_docs)} docs / {len(test_items)} QAs  (answerable={n_ans}, unanswerable={len(test_items)-n_ans})')


## 2. MemSlot saliency — trained + untrained

- **Trained**: short bf16 pass over 120 train contracts (reconstruction + diversity losses, no labels).
- **Untrained**: same architecture, random-init slots, never trained. `word_weights()` works either way because it only relies on the slot→token softmax.

Keeping both lets us test whether MemSlot's training objective actually contributes signal — a core claim of the setup.

In [ ]:
import torch
from deepseek_pipeline import MemSlotSaliency, MemSlotConfig

BATCH = 24 if torch.cuda.get_device_properties(0).total_memory > 30e9 else 8
cfg = MemSlotConfig(n_slots=32, epochs=2, batch_size=BATCH, max_length=512, stride=256)

memslot_trained   = MemSlotSaliency(backbone_name='roberta-base', config=cfg)
memslot_untrained = MemSlotSaliency(backbone_name='roberta-base', config=cfg)

CKPT = str(WORKDIR / 'memslot_trained.pt')
if os.path.exists(CKPT):
    memslot_trained.load(CKPT); print(f'loaded {CKPT}')
else:
    memslot_trained.train_on_contracts(train_contexts[:TRAIN_CAP_FOR_MEMSLOT])
    memslot_trained.save(CKPT); print(f'saved  {CKPT}')

print('untrained MemSlot: slots left at N(0, 0.02) init (no training call)')


## 3. Token budgets — 5× / 10×

One tokenizer (RoBERTa) measures every method's compression so 5×/10× means
the same thing for prune, summary, OCR, and the ablation branches.

In [ ]:
from deepseek_pipeline.metrics import RESOLUTION_MODES, vision_token_count_for_mode

TARGET_RATIOS = [5, 10]
OCR_MODES = ['tiny', 'small', 'base', 'large']

def pick_mode(budget):
    return min(OCR_MODES, key=lambda m: abs(RESOLUTION_MODES[m]['vision_tokens'] - budget))

tokenizer = memslot_trained.tokenizer
for s in test_items:
    s['n_orig'] = len(tokenizer.encode(s['context'], add_special_tokens=False))
    s['budget']   = {R: max(1, s['n_orig'] // R) for R in TARGET_RATIOS}
    s['ocr_mode'] = {R: pick_mode(s['budget'][R]) for R in TARGET_RATIOS}

import pandas as pd
pd.DataFrame([{'n_orig': s['n_orig'],
               'B_5x': s['budget'][5], 'B_10x': s['budget'][10]}
              for s in test_items]).describe().round(0)


## 4. Multi-page A4 rendering — render-then-crop, tier-based

Following the `run.py` pipeline: the full selected passage is rendered
**once** by `render_img_tiered` at a single global font scale. MemSlot's
word weights (already in [0, 1] from its min-max scaling) feed
`assign_tiers(phi, primary_pct=98, secondary_pct=85)` to pick out the 2 %
top-salient words (primary, s_max, topic colour) and the next 13 %
(secondary, interpolated s_mid, dark grey); the rest render at s_min in
light grey. This is the exact 3-tier treatment the existing repo uses
for its distorted-rendering artifact — we just reuse it.

The tall tiered image is then cropped vertically into 1024 × 1448 px A4
tiles (the last tile is white-padded). Each tile is its own OCR call;
decoded text is concatenated in reading order. DeepSeek-OCR resizes any
input to a ~1024² patch grid, so a single A4 tile fits natively while a
long scroll would collapse to noise.

The **text→render→OCR ablation** (`summary-ocr`, `prune-ocr`) uses plain
`render_img` with `s_min = s_max = 22` — uniform font size, no saliency
emphasis. Every pixel of the OCR pipeline is otherwise identical, so a
gap between `memslot-ocr` and `{summary, prune}-ocr` at the same ratio
is attributable to MemSlot's selection, not to OCR being "magic".


In [ ]:
import hashlib, math
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from render.render import render_img, render_img_tiered
from render.utils import assign_tiers

PAGE_W = 1024
PAGE_ASPECT = 1.414              # A4
PAGE_H = int(PAGE_W * PAGE_ASPECT)

def render_pages(word_weights, budget_words, out_dir, tag, uniform=False):
    """Render selected words as one tall image (single global font scale),
    then slice into A4 tiles. Saliency branch reuses the run.py pipeline
    (render_img_tiered + assign_tiers); ablation branch uses render_img
    with s_min == s_max so no word gets emphasized."""
    if not word_weights:
        return []
    B = max(1, min(budget_words, len(word_weights)))
    w_arr = np.array([w for _, w in word_weights], dtype=np.float32)
    idx = (np.sort(np.argpartition(-w_arr, B - 1)[:B])
           if B < len(word_weights) else np.arange(len(word_weights)))
    kept = [word_weights[i] for i in idx]

    if uniform:
        full = render_img(kept, image_width=PAGE_W, s_min=22, s_max=22)
    else:
        phi = w_arr[idx]                     # already in [0, 1] from MemSlot
        tiers = assign_tiers(phi, primary_pct=98.0, secondary_pct=85.0)
        full = render_img_tiered(kept, tiers, image_width=PAGE_W,
                                 s_min=16, s_mid=28, s_max=42)

    W, H = full.size
    n_pages = max(1, math.ceil(H / PAGE_H))
    paths = []
    for p in range(n_pages):
        top = p * PAGE_H
        tile = full.crop((0, top, W, min(top + PAGE_H, H)))
        if tile.size[1] < PAGE_H:             # pad last tile to A4
            bg = Image.new('RGB', (W, PAGE_H), (255, 255, 255))
            bg.paste(tile, (0, 0))
            tile = bg
        path = str(out_dir / f'{tag}_p{p:02d}.png')
        tile.save(path)
        paths.append(path)
    return paths

def uniform_weights(text):
    return [(w, 1.0) for w in text.split()]

RENDERS = WORKDIR / 'renders'
RENDERS.mkdir(exist_ok=True)
ctx_hash = lambda s: hashlib.md5(s.encode()).hexdigest()[:12]


## 5. Compressors — text and OCR branches

In [ ]:
from deepseek_pipeline import DeepSeekOCRCompressor, SaliencyPruner, ApiSummarizer

SUM_PROVIDER = 'deepseek' if os.environ.get('DEEPSEEK_API_KEY') else 'qwen'

ocr        = DeepSeekOCRCompressor()
pruner     = SaliencyPruner(tokenizer)
summarizer = ApiSummarizer(tokenizer, provider=SUM_PROVIDER)
print(f'summarizer provider = {SUM_PROVIDER}')


### 5.a  Build per-document saliency, summaries, and rendered pages

We cache results by `(context_hash, variant)` so each expensive call runs at most once per contract.

In [ ]:
from tqdm.auto import tqdm

w_trained, w_untrained = {}, {}
summary_cache = {}
pages_cache   = {}

for s in tqdm(test_items, desc='per-doc prep'):
    h = ctx_hash(s['context'])
    s['h'] = h

    if h not in w_trained:
        w_trained[h]   = memslot_trained.word_weights(s['context'])
        w_untrained[h] = memslot_untrained.word_weights(s['context'])

    for R in TARGET_RATIOS:
        B_tokens = s['budget'][R]
        B_words  = max(1, int(B_tokens * 0.75))

        for tag, ww in (('ms-tr', w_trained[h]),
                        ('ms-nt', w_untrained[h])):
            key = (h, tag, R)
            if key not in pages_cache:
                pages_cache[key] = render_pages(ww, B_words, RENDERS,
                                                f'{h}_{tag}_r{R}',
                                                uniform=False)

        sk = (h, R)
        if sk not in summary_cache:
            try:
                summary_cache[sk] = summarizer.compress(s['context'], target_tokens=B_tokens, question=None)
            except Exception as e:
                summary_cache[sk] = None
                print(f'summary fail ({h}, R={R}): {type(e).__name__}: {e}')

for s in tqdm(test_items, desc='text→ocr renders'):
    h = s['h']
    for R in TARGET_RATIOS:
        sum_res = summary_cache[(h, R)]
        if sum_res is not None:
            key = (h, 'sum-ocr', R)
            if key not in pages_cache:
                pages_cache[key] = render_pages(uniform_weights(sum_res.text),
                                                budget_words=10**9,
                                                out_dir=RENDERS,
                                                tag=f'{h}_sumocr_r{R}',
                                                uniform=True)

        prune_res = pruner.compress(w_trained[h], keep_ratio=1.0 / R)
        key = (h, 'prn-ocr', R)
        if key not in pages_cache:
            pages_cache[key] = render_pages(uniform_weights(prune_res.text),
                                            budget_words=10**9,
                                            out_dir=RENDERS,
                                            tag=f'{h}_prnocr_r{R}',
                                            uniform=True)

print('pages rendered:', sum(len(v) for v in pages_cache.values()),
      '| unique page variants:', len(pages_cache))

fig, axes = plt.subplots(1, 2, figsize=(12, 8))
sample_h = test_items[0]['h']
for ax, tag in zip(axes, ['ms-tr', 'ms-nt']):
    paths = pages_cache[(sample_h, tag, 5)]
    if paths:
        ax.imshow(Image.open(paths[0]))
    ax.set_title(f'{tag}  page 0 / {len(paths)}  (R=5)')
    ax.axis('off')
plt.tight_layout(); plt.show()


### 5.b  Run every compression method on every test item

In [ ]:
def text_tok_len(t):
    return max(1, len(tokenizer.encode(t, add_special_tokens=False))) if t else 1

def ocr_multipage(paths, mode):
    '''OCR every page and concatenate decoded text; return (text, total_vision_tokens).'''
    if not paths:
        return '', 0
    parts, vis = [], 0
    for p in paths:
        r = ocr.compress(p, mode=mode)
        parts.append(r.decoded_text)
        vis += r.n_vision_tokens
    return '\n\n'.join(parts), vis

all_runs = []
ocr_text_cache = {}  # (h, tag, R) -> (text, n_vis)

def cached_ocr(h, tag, R):
    key = (h, tag, R)
    if key not in ocr_text_cache:
        paths = pages_cache.get(key, [])
        mode = 'base'
        text, vis = ocr_multipage(paths, mode=mode)
        ocr_text_cache[key] = (text, vis, mode, len(paths))
    return ocr_text_cache[key]

for s in tqdm(test_items, desc='compress'):
    h, N = s['h'], s['n_orig']
    base = {'qa_id': s['qa_id'], 'question': s['question'],
            'gold': s['gold'], 'answerable': s['answerable'],
            'n_original_tokens': N}

    all_runs.append({**base, 'method': 'full', 'target_ratio': 1,
                     'context': s['context'], 'n_compressed_tokens': N})

    for R in TARGET_RATIOS:
        B = s['budget'][R]

        pr = pruner.compress(w_trained[h], keep_ratio=1.0 / R)
        all_runs.append({**base, 'method': 'prune', 'target_ratio': R,
                         'context': pr.text, 'n_compressed_tokens': pr.n_tokens})

        sr = summary_cache.get((h, R))
        if sr is not None:
            all_runs.append({**base, 'method': 'summary', 'target_ratio': R,
                             'context': sr.text, 'n_compressed_tokens': sr.n_tokens})

        import random as _r
        rng_local = _r.Random(hash((h, R)) & 0xffffffff)
        words = s['context'].split()
        n_keep = max(1, int(len(words) / R))
        idx = sorted(rng_local.sample(range(len(words)), min(n_keep, len(words))))
        rand_text = ' '.join(words[i] for i in idx)
        all_runs.append({**base, 'method': 'random', 'target_ratio': R,
                         'context': rand_text, 'n_compressed_tokens': text_tok_len(rand_text)})

        words_lead = words[: max(1, int(len(words) / R))]
        lead_text = ' '.join(words_lead)
        all_runs.append({**base, 'method': 'lead', 'target_ratio': R,
                         'context': lead_text, 'n_compressed_tokens': text_tok_len(lead_text)})

        for tag, label in (('ms-tr',   'memslot-ocr-trained'),
                           ('ms-nt',   'memslot-ocr-untrained'),
                           ('sum-ocr', 'summary-ocr'),
                           ('prn-ocr', 'prune-ocr')):
            text, vis, mode, npages = cached_ocr(h, tag, R)
            all_runs.append({**base, 'method': label, 'target_ratio': R,
                             'context': text,
                             'n_compressed_tokens': text_tok_len(text),
                             'n_vision_tokens': vis, 'ocr_mode': mode,
                             'n_pages': npages})

empty_ocr = [r for r in all_runs
             if r['method'].endswith('-ocr') or r['method'].startswith('memslot')]
empty_ocr_bad = sum(1 for r in empty_ocr if not r['context'].strip())
print(f'OCR empty-decode rate: {empty_ocr_bad}/{len(empty_ocr)}')
print('total runs:', len(all_runs))


## 6. QA reader — `qwen-long`

`qwen-long` takes ~10 M tokens so the `full` baseline (up to 36 k) never
truncates. We preflight once before the main loop so a bad key fails in
seconds rather than 500 calls in.

In [ ]:
from deepseek_pipeline import ApiQAReader

reader = (ApiQAReader(provider='qwen-long', model_name='qwen-long')
          if os.environ.get('DASHSCOPE_API_KEY')
          else ApiQAReader(provider='deepseek'))
print('QA reader:', reader.model_name)

probe = max(test_items, key=lambda s: s['n_orig'])
print(f'preflight ({probe["n_orig"]} orig tokens) …')
r = reader.score(probe['context'], probe['question'], probe['gold'])
print(f'  prediction={r.prediction!r}  em={r.em:.2f}  f1={r.f1:.2f}  conf={r.confidence:.2f}')
if r.prediction and probe['context'].lstrip().startswith(r.prediction[:80]):
    raise RuntimeError('Reader echoed the context prefix — likely silent truncation.')


In [ ]:
from tqdm.auto import tqdm
from collections import Counter

for run in all_runs:
    run.setdefault('prediction', ''); run.setdefault('em', float('nan'))
    run.setdefault('f1', float('nan')); run.setdefault('confidence', float('nan'))
    run.setdefault('status', 'pending'); run.setdefault('error_msg', '')

for run in tqdm(all_runs, desc='QA'):
    try:
        res = reader.score(run['context'], run['question'], run['gold'])
        run['prediction'] = res.prediction
        run['em'], run['f1'], run['confidence'] = res.em, res.f1, res.confidence
        run['status'] = 'ok'
    except Exception as e:
        run['status']    = 'api_error'
        run['error_msg'] = f'{type(e).__name__}: {e}'

print('status counts:', dict(Counter(r['status'] for r in all_runs)))


## 7. Evaluation

- `success_rate`: fraction of `status=='ok'` runs — API-health check, not a skill metric.
- `EM_ans / F1_ans`: SQuAD-style on the **answerable** subset of successful runs.
- `abstain_acc`: on unanswerable successes, fraction where the model correctly returned empty.
- `AUPR / P@80R`: CUAD ranking metrics from the reader's self-reported 1-5 confidence.
- `ratio_text`: actual text-token compression (`N_orig / n_compressed_tokens`) — same yardstick for every method.
- `ratio_vis`: for OCR branches, `N_orig / total_vision_tokens`.

In [ ]:
import numpy as np
import pandas as pd
from deepseek_pipeline import cuad_evaluate

df = pd.DataFrame(all_runs)
df['compression_ratio'] = df['n_original_tokens'] / df['n_compressed_tokens'].clip(lower=1)
df['label'] = df.apply(lambda r: 'full' if r['method'] == 'full'
                                 else f"{r['method']}-{r['target_ratio']}x", axis=1)
df.to_csv(WORKDIR / 'cuad_multipage_results.csv', index=False)

def safe_mean(s):
    s = pd.Series(s).dropna()
    return float(s.mean()) if len(s) else float('nan')

rows = []
for label, sub in df.groupby('label'):
    ok = sub[sub['status'] == 'ok']
    ans = ok[ok['answerable']]
    una = ok[~ok['answerable']]
    aupr = p80 = float('nan')
    if len(ok):
        s = cuad_evaluate(ok['prediction'].tolist(), ok['gold'].tolist(),
                          confidences=ok['confidence'].tolist())
        aupr, p80 = s.aupr, s.precision_at_80_recall
    rows.append({
        'label': label,
        'success_rate':       float((sub['status'] == 'ok').mean()),
        'EM_ans':             safe_mean(ans['em']),
        'F1_ans':             safe_mean(ans['f1']),
        'abstain_acc':        float((una['prediction'].astype(str).str.strip() == '').mean()) if len(una) else float('nan'),
        'AUPR':               aupr,
        'P@80R':              p80,
        'ratio_text':         safe_mean(sub['compression_ratio']),
        'ratio_vis':          safe_mean(sub['n_original_tokens'] / sub['n_vision_tokens'].clip(lower=1)) if 'n_vision_tokens' in sub else float('nan'),
        'n_ok':               int((sub['status'] == 'ok').sum()),
        'n_total':            len(sub),
    })
summary = pd.DataFrame(rows).set_index('label').sort_index().round(3)

if 'full' in summary.index and summary.loc['full', 'success_rate'] < 0.9:
    print(f'[WARN] full success_rate={summary.loc["full", "success_rate"]:.2f} — inspect df[df.status!="ok"].')
summary


### Visualizations

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid', context='talk')

plot_df = summary.reset_index().copy()
plot_df['method'] = plot_df['label'].str.replace('-5x', '', regex=False).str.replace('-10x', '', regex=False)
plot_df['ratio']  = plot_df['label'].map(lambda x: 'full' if x == 'full' else x.split('-')[-1])

fig, axes = plt.subplots(2, 2, figsize=(18, 11))
for ax, (metric, title) in zip(axes.ravel(),
        [('F1_ans', 'F1 on answerable'),
         ('abstain_acc', 'Abstention accuracy'),
         ('success_rate', 'API success rate'),
         ('ratio_text', 'Achieved compression ratio')]):
    sns.barplot(data=plot_df, x='method', y=metric, hue='ratio', ax=ax,
                palette={'5x': '#1f77b4', '10x': '#d62728', 'full': '#7f7f7f'})
    ax.set_title(title); ax.set_xlabel(''); ax.set_ylabel(metric)
    for lbl in ax.get_xticklabels():
        lbl.set_rotation(30); lbl.set_ha('right')

plt.suptitle(f'CUAD 5× / 10× | reader={reader.model_name}', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.savefig(WORKDIR / 'multipage_compare.png', dpi=150); plt.show()


In [ ]:
df_ok = df[(df['status'] == 'ok') & df['answerable']].copy()
if len(df_ok):
    pivot = df_ok.pivot_table(index='qa_id', columns='label', values='f1')
    fig, ax = plt.subplots(figsize=(16, max(3, 0.4 * len(pivot))))
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', vmin=0, vmax=1,
                cbar_kws={'label': 'F1'}, ax=ax)
    ax.set_title('Per-QA F1 (answerable, status=ok)')
    ax.set_xlabel(''); ax.set_ylabel('CUAD QA id')
    plt.tight_layout(); plt.savefig(WORKDIR / 'multipage_heatmap.png', dpi=150); plt.show()
else:
    print('No successful answerable runs — heatmap skipped.')


## 8. Artifacts

- `runs/cuad_multipage_results.csv` — per-(method × ratio × QA) predictions, EM/F1, confidences, ratios
- `runs/multipage_compare.png` — aggregate comparison bars
- `runs/multipage_heatmap.png` — per-QA F1 heatmap
- `runs/memslot_trained.pt` — MemSlot checkpoint
- `runs/renders/` — every A4 page seen by OCR

See `EXPERIMENT_DESIGN.md` in the repo root for the full protocol, the rationale
behind each baseline, and a discussion of the 5×/10× metric design.